In [ ]:
import pandas as pd

## Analyse

## 1.1 Parcours d'un employé ayant démissionné contre un employé qui est resté dans l'entreprise,
Note : regarder la classe \"Employee\" pour le code interne de l'analyse. L'approche Notebook se concentre sur la présentation et l'analyse des résulats. Celle permet d'externalise les fonctions de plotting et traitements en présentant juste une interface plus haut niveau pour faire des comparaisons.

In [ ]:
from employee import (
    Employee, COL_MATRICULE, COL_SALARY_EUROS, COL_AGE_YEARS, COL_SENIORITY_GROUP_YEARS,
    COL_CONTRACT_START_YEARS, COL_HAS_RESIGNED, COL_CUSTOM_TIME,
)

dataset_path="Data/RH_dataset.csv"
df_career_analysis:pd.DataFrame=pd.read_csv("Data/RH_dataset.csv", delimiter=";")
df_career_analysis.sort_values(COL_SENIORITY_GROUP_YEARS).groupby(COL_MATRICULE).last().reset_index()
df_career_analysis.dtypes

In [ ]:
# Pensez à changer la seed pour obtenir des séléctions différentes !
# import random
# random.seed(1)
# np.random.seed(1)
# On récupère 2 employées aléatoirement, l'un a démissionné et l'autre non
# employee_resigned = Employee.get_random(df, has_resigned=True, name="Démissionnaire")
# employee_retained = Employee.get_random(df, has_resigned=False, name="Non-démissionnaire")
# Résulat avec la seed 1, hardcodé par matricule pour s'assurer que les explications des figures restent cohérentes

employee_resigned = Employee(df_career_analysis, matricule=216, name="Démissionnaire")
employee_retained = Employee(df_career_analysis, matricule=2057, name="Non-démissionnaire")

print(f"A démissionné   : matricule {employee_resigned.matricule} | {len(employee_resigned.df)} entrées")
print(f"Non-démissionné : matricule {employee_retained.matricule} | {len(employee_retained.df)} entrées")

## Tendance de l'employée n'ayant PAS démissionné
Note : la valeur de CUSTOM_TIME est une version calculée du temps total (en années) passé dans l'entreprise par l'employé. Son calcul se base sur l'ancienneté dans l'entreprise.

In [ ]:
employee_retained.multiplot(x=COL_CUSTOM_TIME)

In [ ]:
employee_retained.compare(employee_resigned, COL_CONTRACT_START_YEARS, show_resignations=True)

On remarque bien sûr que la carrière de l'employé qui est resté aura durée plus longtemps si on prend le temps depuis le début du contrat.

In [ ]:
employee_retained.multiplot(
     others=employee_resigned,
    x=COL_CUSTOM_TIME,
    exclude_cols=[COL_MATRICULE, COL_HAS_RESIGNED, COL_AGE_YEARS],
    show_resignations=True
)

Fait intéressant ici, en regardant Salaire (Euros) et Dernière Augmentation (mois), on peut voir que les deux employés on été augmenté à la même période, mais que l'employé ayant démissionné est payé nettement moins, et son augementation était bien plus faible également. Il a démissionné peu après. Il est plausible que la faiblesse de cette augmentation soit la raison de la démission.

In [ ]:
employee_retained.compare(
    others=employee_resigned,
    column=COL_SALARY_EUROS,
    x=COL_CUSTOM_TIME,
    show_resignations=True
)

## Comparaison par moyenne (cohort) pour discerner des tendances
Faisons une comparaison par moyenne pour discerner des tendances notables entre les carrières démissionnaires et non-démissionnaires. ATTENTION, notre effectif ici est trés variable, il y aura par exemple peu de démissionnaires avec une longue carrière, et la moyenne ne sera donc pas diluée par un grand effectifs.

In [ ]:
Employee.cohort_multiplot(df_career_analysis, order_by=COL_CUSTOM_TIME)

Les démissionnaires forment un profil cohérent : employé plutôt milieu de carrière (35–45 ans), entré récemment, bloqué hiérarchiquement (niveau 1.0–1.5, environ 4 000€), sans augmentation (environ 25 mois) ni promotion (environ 30 mois). Le départ survient quasi systématiquement dans les 18 premiers mois !
Colonnes importantes :
- Dernière augmentation : pic à ~25 mois sans augmentation, variable probablement la plus discriminante
- Dernière promotion : ~30 mois sans évolution au moment du départ
- Parent : pic de démission à Parent=1 vers 0.5 an, possiblement une réévaluation des priorités
- Niveau & Salaire : stagnation totale chez les démissionnaires vs progression continue en poste
- On voit plutôt un problème d'adéquation entre les attentes à l'embauche et la réalité du poste, plutôt qu'un désengagement progressif de l'employé

# 1.2 - Analyse des variables sensibles et biais potentiels
Certaines variables sont sensibles, et risque d'être utilisées comme proxy.
- Âge : variable directement protégée par le Code du travail. Ne peut pas légalement motiver une décision RH.
- Parent : proxy du genre : risque avec les congés parentaux rend cette variable corrélée au sexe féminin. C'est une forte source de biais d'embauche et pour l'évolution de carrière.
- Salaire : proxy du genre et de l'origine via les inégalités salariales structurelles. Encoder le salaire, c'est encoder indirectement ces caractéristiques protégées.
- Niveau hiérarchique : proxy du genre et de l'origine. Les niveaux élevés sont statistiquement moins accessibles à certains groupes, ce biais se retrouve dans la variable  par défaut.
- Ancienneté groupe : proxy de l'âge et du genre. Les interruptions de carrière (congé maternité notamment) créent des zones qui discriminent indirectement.